# Generate the YAML file config input to nextflow for the Bayesian model

This one is for the SV normalized copy numbers
There is no Known filters yet


1. Find all valid files
2. Find all vallid genes
3. Compile a dictionary of options
4. Save the dictionary

In [ ]:
srun  -p componc_cpu --nodes=1 --ntasks-per-node=1 --mem-per-cpu 40G --time=12:00:00 --pty bash -i

In [ ]:
import os
import yaml
import pandas as pd
from glob import glob

# Need load_data
os.chdir("/data1/shahs3/users/salehis/ecdna/src/mixture_model")
from mixture_models_pyro_utils import load_data


batch_dir = '/data1/shahs3/users/salehis/ecdna/pipelines/batches/'
outDir = os.path.join(batch_dir, 'nov_07_bayes_oncogene') ### Where the batch will be saved
os.makedirs(outDir, exist_ok=True)
# TODO: Create one locally  too mkdir -p /Users/salehis/Projects/ecdna/pipelines/batches/oct_13_bayes
_local_path = '/Users/salehis/Projects/ecdna/pipelines/batches/'
print(f"mkdir -p {_local_path}/{os.path.basename(outDir)}")

# Where the outputs of the nextflow will be saved
NF_OUT_DIR = '/data1/shahs3/users/salehis/ecdna/results/bayes_v3_model_nov_07_oncogene/'

# Files that will be actually processed
oncogene_dir = '/data1/shahs3/users/leej39/dlp/analysis/normalization/oncogene_cn_final/'

oncogene_files = glob(os.path.join(oncogene_dir, '*.sv-seg-cn.oncogene.cn.csv'))

files = oncogene_files

# find lx516 files (regardless of case)
#files = [f for f in oncogene_files if 'lx516' in os.path.basename(f).lower()]

len(files)

print(f"Found {len(files)} files to process")

results = None
# Read each file, extract the fname and gene names, then one row per file_gene to the results
for i, f in enumerate(files):
    # if i > 3:
    #     break
    # ignore i < 45
    # if i < 45:
    #     continue
    print(f'{i+1}/{len(files)}: {f}')
    df = pd.read_csv(f, sep=',')
    # all columns except the first one
    genes = df.columns[5:]
    for g in genes:
        try:
            r_counts, tail_vals, r_counts_FULL = load_data(f, locus=g, do_log=True)
        except Exception as e:
            print(f'Error loading {f} {g}: {e}')
            continue
        N_CELLS = len(r_counts)
        if N_CELLS < 30:
            print(f'Skipping {f} {g} because only {N_CELLS} cells')
            continue
        tmp = {}
        tmp['fpath'] = f
        tmp['fname'] = os.path.basename(f)
        # Get the sample name (the first part after split by .)
        tmp['sample'] = os.path.basename(f).split('.')[0]
        # set peakgene or oncogene
        if 'peakgene' in tmp['fname']:
            tmp['type'] = 'peakgene'
        elif 'oncogene' in tmp['fname']:
            tmp['type'] = 'oncogene'
        tmp['gene'] = g
        # Gather some statistics
        # 1. number of cells (i.e., nrows)
        tmp['n_cells'] = df.shape[0]
        # 2. min and max copy number
        tmp['min_cn'] = df[g].min()
        tmp['max_cn'] = df[g].max()
        if results is None:
            results = pd.DataFrame(tmp, index=[0])
        else:
            results = pd.concat([results, pd.DataFrame(tmp, index=[0])])


# Check if they are in the filter file
results['id'] = results['sample'].astype(str) + '__' + results['gene'].astype(str)

orig_shape = results.shape
print(f"Found {results.shape[0]} total sample-gene pairs")

# check for the damn lx516
# is lx516 in sample name (lowercase)
results['sample_lower'] = results['sample'].str.lower()
results_lx516 = results[results['sample_lower'].str.contains('lx516')]
print(f"Found {results_lx516.shape[0]} lx516 sample-gene pairs")

# Save results
results.reset_index(drop=True, inplace=True)
results.to_csv(os.path.join(outDir, 'peakgene_oncogene_genes.csv'), index=True)



In [ ]:
# Check on the results if you want
rsync -azvp \
    iris:/data1/shahs3/users/salehis/ecdna/pipelines/batches/oct_24_bayes_region/*.csv \
    /Users/salehis/Projects/ecdna/test

# Create the yaml file configs 

In [ ]:
# load the resutls, compile into a dictionary and save as yaml
results = pd.read_csv(os.path.join(outDir, 'peakgene_oncogene_genes.csv'), index_col=0)
results['id'] = results['sample'].astype(str) + '__' + results['gene'].astype(str)


config_template_str = """
dataset_name: ''
dat_path: ''
locus: ''
outDir: ''
subset_counts: -1
seed: 100
max_M: 2
holdoff_rate: 0.1
n_steps: 2000
learning_rate: .01
no_loop: true
"""

config_template = yaml.safe_load(config_template_str)

# for each row in the results, copy the template, update the values and save as yaml

res_dict = {}
for i, row in results.iterrows():
    # copy the template
    config = config_template.copy()
    # update the values
    #config['dataset_name'] = f"{row['sample']}_{row['type']}_{row['gene']}"
    config['dataset_name'] = f"{row['sample']}_{row['gene']}"
    config['dat_path'] = row['fpath']
    config['locus'] = row['gene']
    #config['outDir'] = os.path.join(NF_OUT_DIR, row['sample'], row['type'], row['gene'])
    config['outDir'] = os.path.join(NF_OUT_DIR, row['sample'], row['gene'])
    res_dict[config['dataset_name']] = config


# Save 
out_fname = os.path.join(outDir, 'run_configs.yaml')
with open(out_fname, 'w') as f:
    yaml.dump(res_dict, f)


In [ ]:
# read res_dict
with open(out_fname, 'r') as f:
    res_dict = yaml.safe_load(f)

# count the number of keys in res_dict
len(res_dict)
# 147
# 532

In [ ]:
# run nextflow
# 1. add a direcotry under batch
# 2. copy over README.md and run.sh
# 3. sync to the server ./pipelines/sync_cluster.sh 
# 4. on the server, navigate to the batch direcotry and run ./run.sh
# cd /data1/shahs3/users/salehis/ecdna/pipelines/batches/{name}

In [ ]:
screen -r RR

In [ ]:
/data1/shahs3/users/salehis/ecdna/results_summary

In [ ]:
# Pull down the figures
rsync -azvp --relative iris:/data1/shahs3/users/salehis/ecdna/./results/bayes_v3_model_oct_13/**/**/figures/*.p* /Users/salehis/Projects/ecdna
# Pull down the logs    
rsync -azvp --relative iris:/data1/shahs3/users/salehis/ecdna/./results/bayes_v3_model_oct_13/**/**/*.yaml /Users/salehis/Projects/ecdna


rsync -azvp --relative iris:/data1/shahs3/users/salehis/ecdna/./results/bayes_v3_model_oct_13/**/**/figures/*.p* /Users/salehis/Projects/ecdna
# Pull down the logs    
rsync -azvp --relative iris:/data1/shahs3/users/salehis/ecdna/./results/bayes_v3_model_oct_13/**/**/*.yaml /Users/salehis/Projects/ecdna



## Compute and plot the score

In [ ]:
import numpy as np
import seaborn as sns
import scanpy as sc
import os
import pandas as pd
from matplotlib import font_manager
from matplotlib import rcParams
import matplotlib.pyplot as plt
from glob import glob

def setup_dirs(outDir):
    figuresDir = os.path.join(outDir, "figures")
    tablesDir = os.path.join(outDir, "tables")
    dataDir = os.path.join(outDir, "data")
    os.makedirs(dataDir, exist_ok=True)
    os.makedirs(figuresDir, exist_ok=True)
    os.makedirs(tablesDir, exist_ok=True)
    return figuresDir, tablesDir, dataDir

def force_arial():
    arial_font_path = '/home/salehis/projects/cdm/fonts/arial.ttf'
    font_manager.fontManager.addfont(arial_font_path)
    prop = font_manager.FontProperties(fname=arial_font_path)
    print("Arial font forced")

# Set font to Arial
force_arial()
rcParams['font.family'] = 'Arial'
sc.settings.set_figure_params(dpi_save=300)


outDir = '/data1/shahs3/users/salehis/ecdna/results/model_embedding_november_07_2025_ONCO_genes'
figuresDir, tablesDir, dataDir = setup_dirs(outDir)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/model_embedding_october_15_2025/figures/*.p* \
    /Users/salehis/Projects/ecdna/

rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/model_embedding_october_15_2025/tables/*.* \
    /Users/salehis/Projects/ecdna/

### Gather information from the fitted models

In [ ]:
# load all best_mode.yaml unde r/Users/salehis/Projects/ecdna/results/april03_ecdna_gaussian/**/**
os.chdir("/data1/shahs3/users/salehis/ecdna/src/mixture_model")
from compute_bayes_v3_scores_utils import (
    compute_weighted_sum,
    gather_best_model_results,
    compute_metrics,
    plot_scatter,
    make_01,
)

import glob
import os
import yaml
import pandas as pd
from scipy.stats import norm

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler

from mixture_models_pyro_utils import load_data
## Local or on cluster

## UPDATE THIS to NF_OUT_DIR
paths = glob.glob(
    f"{NF_OUT_DIR}/**/**/best_model.yaml",
    recursive=True,
)
res = gather_best_model_results(paths)
res.to_csv(os.path.join(tablesDir, "best_model_results.csv"), index=False)

## Actual analysis

In [ ]:
# plos a scatter plot X: mass_in_window, Y: score
def get_today_str():
    from datetime import date
    today = date.today()
    return today.strftime("%b_%d_%Y").lower()


res = pd.read_csv(os.path.join(tablesDir, 'best_model_results.csv'))

sam = res['sigma_alpha_max'].values.copy()
sam = StandardScaler().fit_transform(sam.reshape(-1, 1)).flatten()

mam = res['mu_alpha_max'].values.copy()
mam = StandardScaler().fit_transform(mam.reshape(-1, 1)).flatten()

res['sam'] = sam
res['mam'] = mam

sd_coeff = .66
#res['score'] = (res['max_alpha']**3) * (q_coeff * res['sigma_alpha_max'] + (1-q_coeff) * res['mu_alpha_max'])
#res['score'] = (res['max_alpha']**4) * (sd_coeff * res['sam'] + (1-sd_coeff) * res['mam'])
res['score'] = (res['max_alpha']**2) * (sd_coeff * res['sam'] + (1-sd_coeff) * res['mam'])
# Normalize between 0 and 1
res['score'] = make_01(res['score'])
res.sort_values(by="score", ascending=False, inplace=True)
res[
    [
        "system_name",
        "locus",
        "max_alpha",
        "mu_alpha_max",
        "sigma_alpha_max",
        "n_cells",
        "ecn",
        "sam",
        "mam",
        "score"
    ]
].head(n=40).reset_index(drop=True)


plot_scatter(
    res,
    plot_col='score',
    min_n_cells=20,
    fpath=os.path.join(figuresDir, f'ecdna_score_vs_id_55{get_today_str()}.pdf'),
    dashed_line=0.55,
)


plot_scatter(
    res,
    plot_col='score',
    min_n_cells=20,
    fpath=os.path.join(figuresDir, f'ecdna_score_vs_id_5{get_today_str()}.pdf'),
    dashed_line=0.5,
)

res.to_csv(os.path.join(tablesDir, f'processed_model_results_{get_today_str()}.csv'), index=False)

# /data1/shahs3/users/salehis/ecdna/results/model_embedding_october_19_2025/tables/processed_model_results.csv

In [ ]:
# check anything that has system_name lx516
res['sn_lower'] = res['system_name'].str.lower().tolist()
res['sn_lower'].value_counts()


uu = res[res['sn_lower'].str.contains('lx516')]
uu['score'].sort_values(ascending=False)




In [ ]:
cases = """NCI-H524_MYC
COLO320DM_MYC
GBM39-DM_EGFR
P-0009535_MDM2
2765_2_MDM2
NCI-H69_GPR75.ASB3
20200721_EGFR
PM_0510_EGFR
20200721_MDM4
P-0009535_CDK4
NCI-H69_MYCN
Lx33_RAPGEF5
P-0009535_EGFR
Lx33_MYC
P-0009535_CCND1
Lx33_CSMD3
P-0093087_CCND1
P-0093087_FGF3
PM_0510_CDK4
Lx298_KLF5
COLO320DM_CDX2
DG1197_AC007131.1
Lx516_DENND6A
Lx516_KDM2A
SPECTRUM-OV-083_TMEM74
DG1134_RP11.66N24.4
SA1180_CCNE1
COLO320-HSR_MYC
COLO320-HSR_CDX2
COLO320-HSR_MCL1
C-K5R275_EGFR
SPECTRUM-OV-036_MCL1
P-0093087_RPL10L
PM_0510_TERT
P-0093087_MDM2
DG1197_ERBB2
"""

# list of cases
cases = cases.split('\n')
cases = list(set(cases))

In [ ]:
from adjustText import adjust_text


#cases = []

# which one has a score over 4 but not in the list
missing = []
for i, row in res.iterrows():
    if row['score'] > 0.4:
        id_ = f"{row['system_name']}_{row['locus']}"
        if id_ not in cases:
            missing.append(id_)


print("Missing cases with score > 0.4:", missing)


# also add anyone whos mass_in_window is close to .5
for i, row in res.iterrows():
    if 0.45 < row['mass_in_window'] < 0.55:
    # mass_in_window between 0.45 and 0.55
    #if (0.4 < row['score'] < 0.6) and (0.5 < row['mass_in_window'] < 0.7):
    #if row['score'] > .55:
        print(row)
        id_ = f"{row['system_name']}_{row['locus']}"
        if id_ not in cases:
            cases.append(id_) 

# Compute log of ncells and use that as size
res['Log(n_cells)'] = np.log2(res['n_cells'] + 1)
res['Max Alpha'] = res['max_alpha']

plt.clf()
fig, ax = plt.subplots(figsize=(7*2, 5*2))
# reduce thinkness of the dashed lines
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.5)
ax.axvline(0.5, color='red', linestyle='--', linewidth=0.5)
# use adjust text to annotate all in the cases list
texts = []
for i, row in res.iterrows():
    if f"{row['system_name']}_{row['locus']}" in cases:
        texts.append(
            ax.text(
                row['mass_in_window'],
                row['score'],
                f"{row['system_name']}_{row['locus']}",
                fontsize=8,
            )
        )
print(f"Annotating {len(texts)} points")
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5), ax=ax)

sns.scatterplot(
    data=res,
    x='mass_in_window',
    y='score',
    hue='Max Alpha',
    size='Log(n_cells)',
    #sizes=(20, 200),
    # size is now in log2 of n_cells
    sizes=(20, 200),
    alpha=0.7,
    ax=ax,
)
ax.set_xlabel('Mass in window around mode')
ax.set_ylabel('eCDNA Score')
# Add a title with the number of points
ax.set_title(f'eCDNA Score vs Mass in window (n={res.shape[0]})')
# Legend outside the plot
# put the legend outside the plot
# set the legend title to 'Max Alpha' and size to 'Log2(N cells)'
# There are two legend titles, one for hue and one for size fix their titles
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(figuresDir, f'ecdna_score_vs_mass_in_window_{get_today_str()}.pdf'))
plt.close()
# Filtered results


In [ ]:
# Show many have score > .5
filtered_res = res[res['score'] > 0.5]
print(f"Number of sample-gene pairs with score > 0.5: {filtered_res.shape[0]}")
filtered_res[

rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/model_embedding_**/figures/*.p* \
    /Users/salehis/Projects/ecdna


In [ ]:
# Plot again
uu_path = '/data1/shahs3/users/leej39/dlp/analysis/modeling/ecdna_model_summary_table_101925.csv'
# cn10 >= 0.01
uu = pd.read_csv(uu_path)
# filter by cn10 >= 0.01
uu = uu[uu['cn10'] >= 0.01].copy()

# Add anyone's whose name has HSR in it, or DM or has score > 0.5
annot_text = []
for i, row in uu.iterrows():
    if ('HSR' in row['individual']) or ('DM' in row['individual']) or (row['score'] > 0.5):
        annot_text.append(f"{row['individual']}_{row['gene']}")

# Just keep the unique ones
annot_text = list(set(annot_text))
print(f"Annotating {len(annot_text)} points")
plt.clf()
fig, ax = plt.subplots(figsize=(7, 5))
# reduce thinkness of the dashed lines
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.5)
ax.axvline(0.5, color='red', linestyle='--', linewidth=0.5)
# use adjust text to annotate all in the cases list
texts = []
for i, row in uu.iterrows():
    if f"{row['individual']}_{row['gene']}" in annot_text:
        texts.append(
            ax.text(
                row['mass_in_window'],
                row['score'],
                f"{row['individual']}_{row['gene']}",
                fontsize=8,
            )
        )

adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5), ax=ax)
sns.scatterplot(
    data=uu,
    x='mass_in_window',
    y='score',
    #hue='Max Alpha',
    #size='Log(n_cells)',
    sizes=(20, 200),
    alpha=0.7,
    # remove the edge color
    edgecolor=None,
    ax=ax,
)
# disable the grid
ax.grid(False)
ax.set_xlabel('Mass in window around mode')
ax.set_ylabel('eCDNA Score')
# Add a title with the number of points
ax.set_title(f'eCDNA Score vs Mass in window (n={uu.shape[0]})')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(figuresDir, f'ecdna_score_vs_mass_in_window_annotated_{get_today_str()}.pdf'))
plt.close()



In [ ]:
import anndata as ad
import numpy as np
adata = ad.read_h5ad('/data1/shahs3/users/salehis/ecdna/results/sv_cna_clones_oct_15/SPECTRUM-OV-052/data/SPECTRUM-OV-052_corrected.h5ad')
# subset to 200
random_state = 42
np.random.seed(random_state)
random_idx = np.random.choice(adata.n_obs, size=200, replace=False)
adata_sub = adata[random_idx, :].copy()
adata_sub.write('/data1/shahs3/users/salehis/ecdna/results/sv_cna_clones_oct_15/SPECTRUM-OV-052/data/SPECTRUM-OV-052_corrected_sub.h5ad')